# Treinos no Google Colab

Guia de execução da sensibilidade do CLAHE (#24) e dos experimentos E1-E4 (#13-#17).
Antes de rodar: **Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU**.

Pré-requisito único feito uma vez na sua máquina: subir um zip de `data/A. Segmentation/` para o Google Drive.

## 1. Montar o Drive e clonar o repositório

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/meowaax/pdi-diabetic-retinopathy.git
%cd pdi-diabetic-retinopathy

## 2. Instalar dependências e preparar os dados

Ajuste o caminho do zip no Drive se for diferente.

In [ ]:
!pip install -q -r requirements.txt

ZIP_NO_DRIVE = '/content/drive/MyDrive/A_Segmentation.zip'  # ajuste se necessário
!mkdir -p data
!unzip -q "$ZIP_NO_DRIVE" -d data/
!ls "data/A. Segmentation"

In [ ]:
!python scripts/build_cache.py

## 3. (opcional) Confirmar que os testes passam no ambiente do Colab

In [ ]:
!python -m pytest -q tests -x

## 4. Apontar checkpoints/logs para o Drive (sobrevive ao fim da sessão)

In [ ]:
import os
RESULTS_NO_DRIVE = '/content/drive/MyDrive/pdi_resultados'
os.makedirs(f'{RESULTS_NO_DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{RESULTS_NO_DRIVE}/logs', exist_ok=True)
os.makedirs(f'{RESULTS_NO_DRIVE}/runs', exist_ok=True)

## 4b. Função auxiliar: roda um comando mostrando a saída em tempo real

`!comando | tail -N` só mostra algo **depois** que o processo termina, o que parece travado em treinos longos. `run_live` imprime cada linha (do loop de treino, uma por época) assim que ela é gerada, com hora e um prefixo identificando a run.

In [ ]:
import subprocess
import sys
import time

def run_live(cmd: list[str], prefix: str = "") -> int:
    """Executa cmd (lista de argumentos) imprimindo stdout/stderr linha a linha, em tempo real, com timestamp. Devolve o código de saída."""
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for line in proc.stdout:
        elapsed = time.strftime('%H:%M:%S', time.gmtime(time.time() - t0))
        print(f"[{elapsed}]{prefix} {line}", end='')
        sys.stdout.flush()
    proc.wait()
    return proc.returncode

## 5. Sensibilidade do CLAHE (issue #24): as 8 configurações

Cada treino usa a divisão de desenvolvimento (42 treino / 12 validação). Roda uma configuração por vez, mostrando o progresso época a época; se a sessão cair, repita a célula — ela pula as configurações que já têm resultado salvo.

**Nota 1:** o log deve mostrar uma linha `mixed_precision=True ignorado: ...`. Isso é esperado: mixed precision foi desativado porque causou `loss=nan` de forma persistente em testes (ver `src/train.py`). O treino roda em precisão total (mais lento, mas correto).

**Nota 2:** `configs/train_default.yaml` usa `micro_batch_size: 2` (o `batch_size: 6` do artigo é só o tamanho lógico; o restante é obtido por gradient accumulation). Se ainda ocorrer `CUDACachingAllocator ... OOM`, adicione `--micro-batch-size 1` no comando abaixo.

In [ ]:
import glob
configs = sorted(glob.glob('configs/clahe_grid/*.yaml'))
print(f'{len(configs)} configurações:')
configs

In [ ]:
import os

for idx, cfg_path in enumerate(configs, start=1):
    run_name = cfg_path.split('/')[-1].replace('.yaml', '')
    out_json = f'results/runs/{run_name}.json'
    prefixo = f' ({idx}/{len(configs)} {run_name})'
    if os.path.exists(out_json):
        print(f'== {idx}/{len(configs)} {run_name}: já existe, pulando ==')
        continue
    print(f'== {idx}/{len(configs)} {run_name}: iniciando ==')
    rc = run_live(
        ['python', 'scripts/run_experiment.py', cfg_path, '--subset', '--run-name', run_name],
        prefix=prefixo,
    )
    print(f'== {idx}/{len(configs)} {run_name}: {"OK" if rc == 0 else f"FALHOU (codigo {rc})"} ==')
    !cp -r results/checkpoints results/logs results/runs "$RESULTS_NO_DRIVE/"

## 6. Conferir a sensibilidade do CLAHE

In [ ]:
!python scripts/collect_results.py --prefix clahe_

## 7. E1-E4: validação cruzada em 5 dobras (issues #13-#17)

Rodar **depois** de decidir o preprocess_mode de cada experimento (E1=rgb, E2=clahe com os parâmetros congelados na #24, E3=green, E4=clahe_green opcional) e os hiperparâmetros finais em `configs/train_default.yaml`.

Exemplo para o E1 (RGB), uma dobra por vez:

In [ ]:
import os

EXPERIMENTO = 'e1_rgb'       # troque para e2_clahe / e3_green / e4_clahe_green
CONFIG = 'configs/train_default.yaml'  # ou um YAML específico do experimento

for k in range(5):
    run_name = f'{EXPERIMENTO}_fold{k}'
    out_json = f'results/runs/{run_name}.json'
    prefixo = f' (dobra {k+1}/5 {run_name})'
    if os.path.exists(out_json):
        print(f'== dobra {k+1}/5 {run_name}: já existe, pulando ==')
        continue
    print(f'== dobra {k+1}/5 {run_name}: iniciando ==')
    rc = run_live(
        ['python', 'scripts/run_experiment.py', CONFIG, '--fold', str(k), '--run-name', run_name],
        prefix=prefixo,
    )
    print(f'== dobra {k+1}/5 {run_name}: {"OK" if rc == 0 else f"FALHOU (codigo {rc})"} ==')
    !cp -r results/checkpoints results/logs results/runs "$RESULTS_NO_DRIVE/"

## 8. Agregar os resultados finais (média ± desvio entre dobras)

In [ ]:
!python scripts/collect_results.py

## 9. Salvar tudo no Drive para devolver depois

`results/runs/*.json` é o essencial (pequeno, texto). Os checkpoints (`.pt`) são grandes; mantenha-os só se for continuar o treino depois, senão descarte para economizar espaço.

In [ ]:
!cp -r results/summaries "$RESULTS_NO_DRIVE/"
!ls -la "$RESULTS_NO_DRIVE"